## Add `med_type` to an asthma NDC spreadsheet

This notebook adds a **`med_type`** column for high-level grouping (**Rescue**, **Controller**, **Montelukast**) from the existing detailed class column **`asthma_med_class_comp`**. It also normalizes selected `asthma_med_class_comp` values and applies a small set of NDC-specific overrides before writing the output workbook.

### Files

- **Input**: `INPUT_XLSX` (Excel) — must include columns `NDC` and `asthma_med_class_comp`.
- **Output**: `OUTPUT_XLSX` (Excel) — same rows/columns as input, with updated `asthma_med_class_comp` and new `med_type` (default output name in the code cell is `Asthma NDCs_5.12.2026_added.xlsx`; change as needed).

Edit `INPUT_XLSX` / `OUTPUT_XLSX` in the code cell to match your filenames.

### Workflow (summary)

1. Trim `NDC` and `asthma_med_class_comp` (so values like `Montelukast ` match `Montelukast`).
2. Rewrite plain `SABA` to `SABA inhaler` in `asthma_med_class_comp`.
3. Map `asthma_med_class_comp` to `med_type` with Rescue → Controller → Montelukast priority (first match wins).
4. Apply NDC-specific overrides for rows that still need manual classification.
5. Rows that still have a **blank** composite class (after trim) get `asthma_med_class_comp` set to **`Non_asthma_med`** and `med_type` set to missing (`<NA>`). Rows already **`Non_asthma_med`** keep `med_type` missing.

### After running

Use the printed `med_type` value counts, or run `python sanity_checker.py` (same folder) to cross-tab `asthma_med_class_comp` × `med_type` on the output workbook.

- **Rescue** — `SABA inhaler`, SABA oral/IV, SAMA, SABA combinations, epinephrine inhaler, OCS, IV methylxanthine (see `rescue_meds` in the code cell).
- **Controller** — ICS/LABA/LAMA and combinations, oral methylxanthine, mast cell stabilizers, non-montelukast LTRA, 5-LOX, biologic (see `controller_meds`).
- **Montelukast** — `Montelukast` only (see `montelukast_meds`).

### NDC-specific overrides

- `51927465600` → `asthma_med_class_comp` = `Montelukast`, `med_type` = `Montelukast`
- `51927137000`, `51927285900`, `52959129301`, `52959146701` → `asthma_med_class_comp` = `SABA_inhaler`, `med_type` = `Rescue`

### Related documentation

See **`README.md`** and **`asthma_medication_approach.pdf`** for the broader medication-classification context.


In [1]:
from pathlib import Path

import pandas as pd

INPUT_XLSX = Path("Asthma NDCs_5.9.2026.xlsx")
OUTPUT_XLSX = Path("Asthma NDCs_5.12.2026_added.xlsx")

rescue_meds = {
    "SABA inhaler",
    "SABA_oral",
    "SABA_iv",
    "Epinephrine_inhaler",
    "SAMA",
    "SABA/SAMA",
    "SABA/ICS",
    "Methylxanthine_iv",
}
controller_meds = {
    "ICS",
    "LABA",
    "LAMA",
    "ICS/LABA",
    "LABA/LAMA",
    "ICS/LABA/LAMA",
    "Methylxanthine_oral",
    "Mast_cell_stabilizers",
    "Non_Montelukast_LTRA",
    "5_LOX",
    "Biologic",
}
montelukast_meds = {"Montelukast"}

data = pd.read_excel(INPUT_XLSX, dtype={"NDC": "string"})
data["NDC"] = data["NDC"].astype("string").str.strip()
# Trim surrounding whitespace so values like "Montelukast " match "Montelukast".
col = data["asthma_med_class_comp"].astype("string").str.strip()
# In this list, plain "SABA" means inhaled SABA (same as "SABA inhaler").
col = col.replace("SABA", "SABA inhaler")
data["asthma_med_class_comp"] = col
# Same order as R case_when (first match wins). Unmatched rows stay NA.
med_type = pd.Series(pd.NA, index=data.index, dtype="string")
med_type.loc[col.isin(rescue_meds)] = "Rescue"
med_type.loc[col.isin(controller_meds) & med_type.isna()] = "Controller"
med_type.loc[col.isin(montelukast_meds) & med_type.isna()] = "Montelukast"
data["med_type"] = med_type

MONTELUKAST_OVERRIDE_NDCS = {"51927465600"}
RESCUE_SABA_INHALER_OVERRIDE_NDCS = {
    "51927137000",
    "51927285900",
    "52959129301",
    "52959146701",
}

montelukast_override = data["NDC"].isin(MONTELUKAST_OVERRIDE_NDCS)
data.loc[montelukast_override, "asthma_med_class_comp"] = "Montelukast"
data.loc[montelukast_override, "med_type"] = "Montelukast"

rescue_override = data["NDC"].isin(RESCUE_SABA_INHALER_OVERRIDE_NDCS)
data.loc[rescue_override, "asthma_med_class_comp"] = "SABA_inhaler"
data.loc[rescue_override, "med_type"] = "Rescue"

comp = data["asthma_med_class_comp"].astype("string")
missing_asthma_class = comp.isna() | comp.str.strip().eq("") | comp.str.strip().str.lower().isin(["nan", "<na>"])
data.loc[missing_asthma_class, "asthma_med_class_comp"] = "Non_asthma_med"
data.loc[missing_asthma_class, "med_type"] = pd.NA

data.to_excel(OUTPUT_XLSX, index=False)
print(f"Wrote {OUTPUT_XLSX.resolve()} ({len(data)} rows).")
print(data["med_type"].value_counts(dropna=False))

Wrote C:\Users\shiroa1\OneDrive - VUMC\inhaler_helper_functions\Asthma NDCs_5.12.2026_added.xlsx (23469 rows).
med_type
<NA>           20080
Controller      1442
Rescue          1437
Montelukast      510
Name: count, dtype: Int64
